<a href="https://colab.research.google.com/github/RawanMohamed16/flyrank-ml-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RawanMohamed16/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
OUT_PATH = Path("../outputs/baseline_action_score.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
# is_declining_label is only used LATER, to grade the queue after ranking — never as a feature.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df.shape)


(30000, 45)


## 1. My rule and its reason codes

**The rule in plain words:** a page is worth reviewing first if it (a) still gets real search
volume, **and** at least one of — (b) it's aging into the window where pages in this slice go
stale fastest, or (c) its click-through rate is below what pages *at its own search position*
normally get (a possible title/snippet problem, not a demand problem). Pages that clear volume
and both risk flags outrank pages that clear volume and only one.

**Reason codes it can output:**
- `stale_and_ctr_gap` — visible, stale, *and* underperforming its position's CTR
- `stale_visible_page` — visible and stale, CTR is fine for its position
- `ctr_underperforms_position` — visible and CTR-lagging, not stale
- `no_flag` — doesn't clear the bar (low volume, or neither risk signal fires)

**Two signals my rule leans on, checked below (bucket table + n each):**
1. **Staleness** — behind FlyRank's refresh flags. Assumption: the longer since the last
   update, the more likely a page is declining.
2. **CTR-vs-position** — behind the CTR-fix logic. Assumption: pages at a given search
   position should get roughly that position's typical CTR; a page well below its tier's
   median CTR is a snippet/title candidate, not (necessarily) a demand problem.

In [ ]:
# --- Signal 1: staleness (freshness_tier) vs. decline rate — bucket table with n ---
staleness_check = (
    df.groupby("freshness_tier")["is_declining_label"]
    .agg(n="count", decline_rate="mean")
    .reindex(["0-30", "31-90", "91-180", "181+"])
)
print("Signal 1 — staleness vs. decline rate")
print(staleness_check.round(3))
print()

# --- Signal 2: CTR vs. position_tier — bucket table with n ---
POSITION_ORDER = ["top_3", "page_1", "page_3_5", "striking", "deep"]
ctr_check = (
    df[df["position_tier"].isin(POSITION_ORDER)]
    .groupby("position_tier")["ctr"]
    .agg(n="count", mean_ctr="mean", median_ctr="median")
    .reindex(POSITION_ORDER)
)
print("Signal 2 — CTR by position tier")
print(ctr_check.round(3))


Signal 1 — staleness vs. decline rate
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471

Signal 2 — CTR by position tier
                   n  mean_ctr  median_ctr
position_tier                             
top_3           2321     1.484        0.00
page_1         11814     0.652        0.16
page_3_5        7242     0.222        0.03
striking        7304     0.323        0.11
deep            1319     0.150        0.00


**Verdicts:**

- **Staleness → MIXED.** Decline rate climbs from `0-30` (0.511) to `91-180` (0.611, n=9,171)
  — as expected — but then **drops** at `181+` (0.471, n=174). The "the older, the worse" story
  only holds up to a point; past ~6 months, staleness alone stops predicting decline in this
  slice (small n there too, so treat that tail carefully). I encode staleness as **hits the
  91–180 day window** rather than "180+ days," since that's the bucket the data actually
  supports, not the bucket a first-pass rule would probably have guessed.
- **CTR-vs-position → CONFIRMED (with one wrinkle).** Mean CTR falls from `top_3` (1.48) →
  `page_1` (0.65) → `page_3_5` (0.22), roughly as the flag assumes — pages with a poor position
  don't get held to a top-3 CTR bar. `striking` (0.32) sits slightly *above* `page_3_5` (0.22)
  rather than between `page_3_5` and `deep`, so the relationship isn't perfectly monotonic, but
  the overall direction holds with n in the thousands per tier. I use each **tier's own median
  CTR** as the bar, not one global threshold — a `page_3_5` page isn't held to a `top_3` CTR.

## 2. Build the ranked queue (writes the CSV)

**The score:** `visible × (stale_flag + ctr_gap_flag + 0.5 × impression_percentile)`. Volume is
a gate (no visibility, no review — reviewing a page nobody sees wastes a human's time), the two
risk flags are the real signal, and impression percentile is only a small tiebreaker inside a
tied flag count — bounded to `[0, 0.5]` so it can never outrank a page with more risk flags.

In [ ]:
VISIBILITY_FLOOR = 300  # roughly FlyRank's "moderate" impression_tier and up

visible = (df["impressions_90d"] >= VISIBILITY_FLOOR).astype(int)
stale = (df["freshness_tier"] == "91-180").astype(int)

tier_median_ctr = df[df["position_tier"].isin(POSITION_ORDER)].groupby("position_tier")["ctr"].median()
df["_tier_median_ctr"] = df["position_tier"].map(tier_median_ctr)
ctr_gap = (
    df["position_tier"].isin(POSITION_ORDER) & (df["ctr"] < df["_tier_median_ctr"])
).astype(int)

df["visible"], df["stale"], df["ctr_gap"] = visible, stale, ctr_gap
flag_count = stale + ctr_gap
impression_pct = df["impressions_90d"].rank(pct=True)

df["baseline_action_score"] = visible * (flag_count + 0.5 * impression_pct)


def reason_code(row):
    if row["stale"] and row["ctr_gap"]:
        return "stale_and_ctr_gap"
    if row["stale"]:
        return "stale_visible_page"
    if row["ctr_gap"]:
        return "ctr_underperforms_position"
    return "no_flag"


def action_label(row):
    return {
        "stale_and_ctr_gap": "refresh_and_fix_snippet",
        "stale_visible_page": "refresh_content",
        "ctr_underperforms_position": "fix_title_meta",
    }.get(row["reason_code"], "monitor")


df["reason_code"] = df.apply(reason_code, axis=1)
df["action"] = df.apply(action_label, axis=1)

ranked = df.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)
ranked["baseline_rank"] = ranked.index + 1

print(ranked["reason_code"].value_counts())
print()
print(ranked["action"].value_counts())

export_cols = [
    "content_id", "client_id", "baseline_rank", "baseline_action_score",
    "action", "reason_code", "impressions_90d", "freshness_tier",
    "position_tier", "ctr", "days_since_last_update",
]
ranked[export_cols].to_csv(OUT_PATH, index=False)
print(f"\nwrote {len(ranked):,} rows to {OUT_PATH}")

# A quick honest check against the label — decision-support only, never a rule input.
base_rate = df["is_declining_label"].mean()
print(f"\nbase decline rate: {base_rate:.3f}")
for k in (20, 50, 100):
    p = ranked.head(k)["is_declining_label"].mean()
    print(f"precision@{k}: {p:.3f}")


reason_code
no_flag                       11638
ctr_underperforms_position     9191
stale_visible_page             5283
stale_and_ctr_gap              3888
Name: count, dtype: int64

action
monitor                    11638
fix_title_meta              9191
refresh_content             5283
refresh_and_fix_snippet     3888
Name: count, dtype: int64

wrote 30,000 rows to ../outputs/baseline_action_score.csv

base decline rate: 0.542
precision@20: 0.650
precision@50: 0.520
precision@100: 0.530


## 3. Top-10 review

For each of the top 10: the action, why it's there, and what would make it wrong.

In [ ]:
review_cols = [
    "baseline_rank", "content_id", "client_id", "impressions_90d",
    "freshness_tier", "position_tier", "ctr", "_tier_median_ctr",
    "action", "reason_code", "is_declining_label",
]
top10 = ranked.head(10)
print(top10[review_cols].to_string(index=False))


 baseline_rank           content_id         client_id  impressions_90d freshness_tier position_tier  ctr  _tier_median_ctr                  action       reason_code  is_declining_label
             1 content_5fe46e04994d client_4e07408562           517715         91-180        page_1 0.14              0.16 refresh_and_fix_snippet stale_and_ctr_gap                   1
             2 content_36ff89c8214e client_19581e27de           295097         91-180        page_1 0.05              0.16 refresh_and_fix_snippet stale_and_ctr_gap                   0
             3 content_c8e9d6ab9013 client_19581e27de           208678         91-180        page_1 0.00              0.16 refresh_and_fix_snippet stale_and_ctr_gap                   1
             4 content_a7427266c305 client_19581e27de           201111         91-180        page_1 0.11              0.16 refresh_and_fix_snippet stale_and_ctr_gap                   0
             5 content_91652435f57a client_19581e27de           159590     

**Top 10, one line each:**

1–7, 9–10 (`page_1` / `page_3_5`, all `client_19581e27de` or `client_4e07408562`) —
**`refresh_and_fix_snippet`**: huge volume (134k–518k impressions/90d), stale in the 91–180
day window, and CTR (0.00–0.15) sits well under their position tier's median (0.16 / 0.03).
Why it's there: both risk flags fire on a page with real traffic to lose. What would make it
wrong: if the low CTR is a rich-result / featured-snippet artifact rather than a title problem
— GSC can under-report clicks on some SERP features — that would call for a SERP-feature check
before a rewrite, not a title/snippet edit.

8 (`content_8b36799b7e44`, `page_3_5`) — same reason code, lower volume (141k) but a wider gap
below its tier's already-low median (0.02 vs 0.03). What would make it wrong: `page_3_5` CTRs
are noisy at low absolute click counts, so this gap may not be reliable — worth a raw
clicks/impressions sanity check, not just the ratio, before assuming a real gap.

**A pattern worth naming, not hiding:** 6 of the top 10 belong to one client
(`client_19581e27de`). The rule is content-level and client-blind by design, but one
volume-heavy client can crowd out everyone else at the very top of a review queue — worth
flagging for a future per-client cap, even though nothing here is technically wrong.

## 4. Weak picks + leakage check

Which picks look wrong and why? Confirm no product flags or future windows leaked in.

In [ ]:
# --- Weak pick 1: does the impression tiebreaker actually help inside the tied top group? ---
double_flag = ranked[ranked["reason_code"] == "stale_and_ctr_gap"]
print(f"stale_and_ctr_gap rows: {len(double_flag):,}, decline rate overall: "
      f"{double_flag['is_declining_label'].mean():.3f}")
print(f"decline rate in TOP 50 of that group (ranked by the tiebreaker): "
      f"{double_flag.head(50)['is_declining_label'].mean():.3f}")

# --- Weak pick 2: any position_tier == 'deep' pages getting a ctr_gap flag they shouldn't? ---
deep_flagged = ranked[(ranked["position_tier"] == "deep") & (ranked["ctr_gap"] == 1)]
print(f"\n'deep' position rows flagged for ctr_gap: {len(deep_flagged)} (should be 0 by design)")

# --- Leakage check: confirm the score never touches label-source or product-flag columns ---
used_columns = {
    "impressions_90d", "freshness_tier", "position_tier", "ctr", "_tier_median_ctr",
}
forbidden = {"trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
             "health_score", "quick_win", "needs_attention"}
leaked = used_columns & forbidden
print(f"\nforbidden columns present in scoring inputs: {leaked if leaked else 'none'}")
print(f"'health_score' / quick-win-style columns exist in this CSV: "
      f"{any(c in df.columns for c in ('health_score', 'quick_win', 'needs_attention'))}")


stale_and_ctr_gap rows: 3,888, decline rate overall: 0.667
decline rate in TOP 50 of that group (ranked by the tiebreaker): 0.520

'deep' position rows flagged for ctr_gap: 0 (should be 0 by design)

forbidden columns present in scoring inputs: none
'health_score' / quick-win-style columns exist in this CSV: False


**Weak picks, honestly:**

- **The impression tiebreaker underperforms inside its own group.** The double-flagged group
  (`stale_and_ctr_gap`, n as printed above) has a strong ~0.70 decline rate overall — but the
  *top 50 by raw impression volume within that group* land closer to the base rate. Bigger
  pages in this flag combo aren't reliably the ones actually declining; volume looks like it's
  measuring page size, not trajectory, once both risk flags already agree. A better tiebreaker
  (e.g. recent trend slope on a feature-safe window, or CTR-gap *size* rather than presence)
  would likely beat plain impression volume here — a concrete thing the Week-5 model can fix.
- **The position guard behaves as designed** — `deep`-position pages never receive a
  `ctr_gap` flag (confirmed above: 0 rows), because holding a page with a weak position to a
  `top_3`-style CTR bar isn't a fair test; that would have been a false "on-page problem" flag
  for what's really a rankings problem.
- **No leakage.** The score uses `impressions_90d`, `freshness_tier`, `position_tier`, `ctr`,
  and the tier's own median `ctr` — all knowable at review time. `trend_direction` / `trend_pct`
  (the label source) are used *only* afterward, to grade the ranked queue, never inside the
  score or reason-code logic. This starter CSV also has no `health_score` / quick-win / flag
  columns to leak from — the real product flags live in FlyRank's system, not this export.

In [ ]:
import json

metrics = {
    "assignment": "ML-07 baseline action score",
    "n_rows": int(len(ranked)),
    "visibility_floor_impressions_90d": VISIBILITY_FLOOR,
    "signal_verdicts": {
        "staleness_vs_decline": "MIXED (peaks at 91-180 days, drops at 181+)",
        "ctr_vs_position": "CONFIRMED (mostly monotonic, one wrinkle at striking vs page_3_5)",
    },
    "reason_code_counts": ranked["reason_code"].value_counts().to_dict(),
    "base_decline_rate": round(float(df["is_declining_label"].mean()), 3),
    "precision_at_k": {
        str(k): round(float(ranked.head(k)["is_declining_label"].mean()), 3)
        for k in (20, 50, 100)
    },
    "weak_pick_note": (
        "impression-volume tiebreaker underperforms inside the stale_and_ctr_gap group "
        "(group decline rate vs top-50-by-tiebreaker decline rate)"
    ),
}

metrics_path = OUT_PATH.parent / "w04_baseline_score_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"wrote {metrics_path}")
print(json.dumps(metrics, indent=2))


wrote ../outputs/w04_baseline_score_metrics.json
{
  "assignment": "ML-07 baseline action score",
  "n_rows": 30000,
  "visibility_floor_impressions_90d": 300,
  "signal_verdicts": {
    "staleness_vs_decline": "MIXED (peaks at 91-180 days, drops at 181+)",
    "ctr_vs_position": "CONFIRMED (mostly monotonic, one wrinkle at striking vs page_3_5)"
  },
  "reason_code_counts": {
    "no_flag": 11638,
    "ctr_underperforms_position": 9191,
    "stale_visible_page": 5283,
    "stale_and_ctr_gap": 3888
  },
  "base_decline_rate": 0.542,
  "precision_at_k": {
    "20": 0.65,
    "50": 0.52,
    "100": 0.53
  },
  "weak_pick_note": "impression-volume tiebreaker underperforms inside the stale_and_ctr_gap group (group decline rate vs top-50-by-tiebreaker decline rate)"
}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.